In [4]:
!pip install -q mlflow
import os
import sys
import time
import shutil
import subprocess
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
import mlflow
from mlflow import MlflowClient

subprocess.run(["pkill", "-f", "mlflow"], capture_output=True)
time.sleep(2)
for path in ["mlflow.db"]:
    if os.path.exists(path):
        os.remove(path)
for folder in ["mlruns", "mlartifacts"]:
    shutil.rmtree(folder, ignore_errors=True)

print("MLflow version:", mlflow.__version__)

np.random.seed(42)
number_of_orders = 500
df = pd.DataFrame()
df["distance_km"] = np.random.uniform(1, 15, number_of_orders)
df["items"] = np.random.randint(1, 10, number_of_orders)
df["hour"] = np.random.randint(8, 23, number_of_orders)
noise = np.random.normal(0, 3, number_of_orders)
df["delivery_minutes"] = 10 + 3 * df["distance_km"] + 1.5 * df["items"] + noise

X = df[["distance_km", "items", "hour"]]
y = df["delivery_minutes"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training rows:", len(X_train))
print("Test rows    :", len(X_test))
print(df.head())

for depth in [2, 5, 10]:
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    mae = mean_absolute_error(y_test, model.predict(X_test))
    print("max_depth =", depth, "   MAE =", round(mae, 2))

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("P06-delivery")
print("Tracking URI:", mlflow.get_tracking_uri())

with mlflow.start_run(run_name="first-tree"):
    depth = 4
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    mae = mean_absolute_error(y_test, model.predict(X_test))
    mlflow.log_param("max_depth", depth)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_metric("mae_minutes", mae)
    mlflow.set_tag("student", "demo")
print("Run recorded. MAE =", round(mae, 2))

runs = mlflow.search_runs(experiment_names=["P06-delivery"])
print(runs[["tags.mlflow.runName", "params.max_depth", "metrics.mae_minutes"]])

for depth in [1, 2, 3, 5, 8, 12]:
    with mlflow.start_run(run_name="tree-depth-" + str(depth)):
        model = DecisionTreeRegressor(max_depth=depth, random_state=42)
        model.fit(X_train, y_train)
        mae = mean_absolute_error(y_test, model.predict(X_test))
        mlflow.log_param("max_depth", depth)
        mlflow.log_metric("mae_minutes", mae)
    print("max_depth =", depth, "   MAE =", round(mae, 2))

best = mlflow.search_runs(
    experiment_names=["P06-delivery"],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.mae_minutes ASC"],
    max_results=1,
)
best_run_name = best["tags.mlflow.runName"][0]
best_depth = int(best["params.max_depth"][0])
best_mae = best["metrics.mae_minutes"][0]
print("Best run  :", best_run_name)
print("Best depth:", best_depth)
print("Best MAE  :", round(best_mae, 2))

with mlflow.start_run(run_name="winner-saved"):
    model = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
    model.fit(X_train, y_train)
    mae = mean_absolute_error(y_test, model.predict(X_test))
    mlflow.log_param("max_depth", best_depth)
    mlflow.log_metric("mae_minutes", mae)
    mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        registered_model_name="delivery-time-model",
        serialization_format="cloudpickle",
    )
print("Model saved and registered. MAE =", round(mae, 2))

client = MlflowClient()
for v in client.search_model_versions("name='delivery-time-model'"):
    print("Version:", v.version, "   Created by run:", v.run_id)

loaded_model = mlflow.sklearn.load_model("models:/delivery-time-model/1")
new_order = pd.DataFrame({"distance_km": [8], "items": [3], "hour": [19]})
print("Predicted delivery time:", round(loaded_model.predict(new_order)[0], 1), "minutes")

cancer = load_breast_cancer(as_frame=True)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=42, stratify=cancer.target
)

mlflow.set_experiment("cancer-diagnosis")


def track_run(run_name, model, params, tags):
    with mlflow.start_run(run_name=run_name):
        model.fit(Xc_train, yc_train)
        predictions = model.predict(Xc_test)
        accuracy = accuracy_score(yc_test, predictions)
        f1 = f1_score(yc_test, predictions)
        mlflow.log_params(params)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("f1_score", f1)
        mlflow.set_tags(tags)
        info = mlflow.sklearn.log_model(
            sk_model=model,
            name="model",
            serialization_format="cloudpickle",
            input_example=Xc_train.head(2),
        )
        mlflow.set_tag("model_uri", info.model_uri)
    print(f"{run_name:<22} accuracy = {accuracy:.4f}   F1 = {f1:.4f}")


track_run(
    "logreg-baseline",
    LogisticRegression(max_iter=10000),
    {"model_type": "LogisticRegression", "max_iter": 10000},
    {"model": "logreg"},
)

for n_estimators in [10, 50, 100]:
    for max_depth in [3, 5, 10]:
        track_run(
            f"rf-n{n_estimators}-d{max_depth}",
            RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42),
            {"model_type": "RandomForest", "n_estimators": n_estimators, "max_depth": max_depth},
            {"sweep": "rf"},
        )

all_runs = mlflow.search_runs(
    experiment_names=["cancer-diagnosis"],
    order_by=["metrics.f1_score DESC"],
)
print(all_runs[["tags.mlflow.runName", "metrics.accuracy", "metrics.f1_score"]].to_string(index=False))

best = mlflow.search_runs(
    experiment_names=["cancer-diagnosis"],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.f1_score DESC", "metrics.accuracy DESC"],
    max_results=1,
)
best_name = best["tags.mlflow.runName"][0]
best_f1 = best["metrics.f1_score"][0]
best_accuracy = best["metrics.accuracy"][0]
print("Best run:", best_name, "  F1 =", round(best_f1, 4), "  accuracy =", round(best_accuracy, 4))

registered = mlflow.register_model(best["tags.model_uri"][0], "cancer-classifier")
print("Registered cancer-classifier version", registered.version)

cancer_model = mlflow.sklearn.load_model("models:/cancer-classifier/1")
first_five = Xc_test.head(5)
predicted = cancer_model.predict(first_five)
print(pd.DataFrame({
    "predicted": [cancer.target_names[p] for p in predicted],
    "actual": [cancer.target_names[a] for a in yc_test.head(5)],
}, index=first_five.index))

baseline = all_runs[all_runs["tags.mlflow.runName"] == "logreg-baseline"].iloc[0]
rf_runs = all_runs[all_runs["tags.sweep"] == "rf"]
best_rf = rf_runs.iloc[0]
print()
print(
    f"The best model was {best_name}, with an F1-score of {best_f1:.4f} and accuracy of {best_accuracy:.4f}. "
    f"The Logistic Regression baseline reached F1 = {baseline['metrics.f1_score']:.4f} (accuracy {baseline['metrics.accuracy']:.4f}), "
    f"while the best Random Forest ({best_rf['tags.mlflow.runName']}) reached F1 = {best_rf['metrics.f1_score']:.4f} "
    f"(accuracy {best_rf['metrics.accuracy']:.4f}). "
    + ("So the Random Forest sweep improved on the baseline."
       if best_rf["metrics.f1_score"] > baseline["metrics.f1_score"]
       else "So the Random Forest sweep did not improve on the baseline; the simpler Logistic Regression was as good or better.")
)

ui_process = subprocess.Popen([
    sys.executable, "-m", "mlflow", "ui",
    "--backend-store-uri", "sqlite:///mlflow.db",
    "--host", "0.0.0.0",
    "--port", "5001",
    "--allowed-hosts", "*",
    "--cors-allowed-origins", "*",
])
time.sleep(15)
try:
    from google.colab.output import eval_js
    print("MLflow UI:", eval_js("google.colab.kernel.proxyPort(5001)"))
except ImportError:
    print("MLflow UI: http://127.0.0.1:5001")

MLflow version: 3.18.0
Training rows: 400
Test rows    : 100
   distance_km  items  hour  delivery_minutes
0     6.243562      6    18         36.064045
1    14.310000      1    21         55.905380
2    11.247915      9    14         58.715004
3     9.381219      1     8         37.832316
4     3.184261      5     9         26.350811
max_depth = 2    MAE = 4.59
max_depth = 5    MAE = 3.31
max_depth = 10    MAE = 4.07


2026/11/13 15:04:12 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/13 15:04:12 INFO mlflow.store.db.utils: Updating database tables
2026/11/13 15:04:15 INFO mlflow.tracking.fluent: Experiment with name 'P06-delivery' does not exist. Creating a new experiment.


Tracking URI: sqlite:///mlflow.db
Run recorded. MAE = 3.47
  tags.mlflow.runName params.max_depth  metrics.mae_minutes
0          first-tree                4             3.465814
max_depth = 1    MAE = 6.11
max_depth = 2    MAE = 4.59
max_depth = 3    MAE = 4.04
max_depth = 5    MAE = 3.31
max_depth = 8    MAE = 3.96


2026/11/13 15:04:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


max_depth = 12    MAE = 4.39
Best run  : tree-depth-5
Best depth: 5
Best MAE  : 3.31


Successfully registered model 'delivery-time-model'.
Created version '1' of model 'delivery-time-model'.
2026/11/13 15:04:21 INFO mlflow.tracking.fluent: Experiment with name 'cancer-diagnosis' does not exist. Creating a new experiment.


... (19 outputs left out)

2026/11/13 15:04:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/usr/local/lib/python3.13/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling 

rf-n100-d10            accuracy = 0.9561   F1 = 0.9655
tags.mlflow.runName  metrics.accuracy  metrics.f1_score
    logreg-baseline          0.964912          0.972603
        rf-n100-d10          0.956140          0.965517
         rf-n100-d5          0.956140          0.965517
         rf-n100-d3          0.956140          0.965517
         rf-n50-d10          0.956140          0.965517
          rf-n50-d5          0.956140          0.965517
          rf-n10-d5          0.956140          0.965517
          rf-n50-d3          0.947368          0.958333
         rf-n10-d10          0.938596          0.951049
          rf-n10-d3          0.938596          0.951049
Best run: logreg-baseline   F1 = 0.9726   accuracy = 0.9649
Registered cancer-classifier version 1
     predicted     actual
256  malignant  malignant
428     benign     benign
501  malignant  malignant
363     benign     benign
564  malignant  malignant

The best model was logreg-baseline, with an F1-score of 0.9726 and accura

Successfully registered model 'cancer-classifier'.
Created version '1' of model 'cancer-classifier'.


MLflow UI: https://5001-m-s-usw3b4-jl7szdmrkki8-b.us-west3.prod.r.colab.dev
